# Lab 25-11 · Test Cost, Burn-In and Guard Bands

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import optimize
import testcost as t

## Part 1 · Four sites or eight?

*Predict* the eight-site efficiency that matches four sites at 90%.

In [2]:
c4 = t.cost_per_die(250, 2.0, 4, 0.90, 0.3)
eta8 = optimize.brentq(lambda e: t.cost_per_die(250, 2.0, 8, e, 0.3) - c4, 0.1, 1)
print(f"four sites: ${c4:.4f} per die; eight sites break even at efficiency {eta8:.2f}")
ns = np.arange(1, 17)
plt.figure()
for e in (0.7, 0.8, 0.9, 0.97):
    plt.plot(ns, [t.cost_per_die(250, 2.0, n, e, 0.3) for n in ns], label=f"efficiency {e}")
plt.xlabel("sites"); plt.ylabel("cost per die ($)"); plt.legend(); plt.grid(True)
plt.show()

four sites: $0.0503 per die; eight sites break even at efficiency 0.75


## Part 2 · Burn-in

*Predict* the fraction of the β = 0.3 weak population removed by 1,200 use hours.

In [3]:
f = t.weak_fail(1200, 100, 0.3)
mc = t.simulate_burn_in(0.01, 100, 0.3, 1200)
print(f"closed form {f:.4f}, Monte Carlo {mc:.4f}")
hrs = np.logspace(1, 5, 80)
plt.figure()
for b in (0.3, 0.5, 0.8):
    plt.semilogx(hrs, [t.weak_fail(h, 100, b) for h in hrs], label=f"beta {b}")
plt.axvline(1200, ls=":", color="k")
plt.xlabel("use-equivalent hours"); plt.ylabel("weak fraction failed"); plt.legend(); plt.grid(True)
plt.show()

closed form 0.8785, Monte Carlo 0.8810


## Part 3 · Guard band

*Predict* the guard band for 100 ppm escapes.

In [4]:
g = optimize.brentq(lambda g: t.escapes_overkill(3.0, g, 3.0, 0.15, 0.02)[0] - 1e-4, 0, 0.1)
print(f"guard band {1000*g:.1f} MHz, overkill {100*t.escapes_overkill(3.0, g, 3.0, 0.15, 0.02)[1]:.1f} %")
print("40 MHz, integral:", t.escapes_overkill(3.0, 0.04, 3.0, 0.15, 0.02))
print("40 MHz, Monte Carlo:", t.simulate_escapes(3.0, 0.04, 3.0, 0.15, 0.02))
gs = np.linspace(0, 0.08, 40)
eo = [t.escapes_overkill(3.0, x, 3.0, 0.15, 0.02) for x in gs]
plt.figure()
plt.semilogy(1000 * gs, [1e6 * e for e, _ in eo], label="escapes")
plt.semilogy(1000 * gs, [1e6 * o for _, o in eo], label="overkill")
plt.xlabel("guard band (MHz)"); plt.ylabel("ppm"); plt.legend(); plt.grid(True)
plt.show()

guard band 50.4 MHz, overkill 13.1 %
40 MHz, integral: (0.00045078576602087823, 0.10468697688748606)
40 MHz, Monte Carlo: (0.0004605, 0.1048555)


## Part 4 · Process spread

*Predict* how bin revenue changes when σ falls from 0.15 to 0.10 GHz.

In [5]:
for s in (0.15, 0.10):
    print(f"sigma {s}: fractions {[round(x, 4) for x in t.bin_fractions(3.0, s, [3.2, 3.0, 2.8])]}, "
          f"revenue ${t.bin_revenue(3.0, s, [3.2, 3.0, 2.8], [500, 400, 300, 0]):.1f}")

sigma 0.15: fractions [np.float64(0.0912), np.float64(0.4088), np.float64(0.4088), np.float64(0.0912)], revenue $331.8
sigma 0.1: fractions [np.float64(0.0228), np.float64(0.4772), np.float64(0.4772), np.float64(0.0228)], revenue $345.4
